# Diabetes Risk Analysis: The Power of Lifestyle
## Executive Summary

After an in-depth analysis of the dataset (Data Storytelling & Multivariate Analysis), we uncovered 3 critical insights:

1. **Lifestyle Beats Genetics:** Patients with a family history of diabetes who maintain an excellent quality of life have a lower risk than those without a family history who live poorly. Exercise, Diet, and Stress levels are the primary drivers.
2. **The Gender Myth:** Gender plays absolutely no role in increasing diabetes risk within this dataset.
3. **The Age Paradox:** Although overall risk increases with age, young patients categorized as "High Risk" exhibit much more aggressive (higher) fasting blood sugar levels than elderly patients in the same category.


## 1. Data Import & Preparation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("diabetes_risk_prediction_dataset.csv")


In [ ]:
# Data Cleaning (Imputation)
df['Physical_Activity_Level'] = df['Physical_Activity_Level'].fillna('Unknown')
df['Medication_Adherence'] = df['Medication_Adherence'].fillna('Unknown')

df['Exercise_Hours_Per_Week'] = df['Exercise_Hours_Per_Week'].fillna(
    df.groupby('Physical_Activity_Level')['Exercise_Hours_Per_Week'].transform('median')
)

remaining_metric = ['Age', 'Height_cm', 'Weight_kg', 'Blood_Glucose', 'HbA1c', 
                    'Total_Cholesterol', 'HDL', 'LDL', 'Triglycerides', 
                    'Daily_Walking_Minutes', 'Sleep_Hours']
for col in remaining_metric:
    df[col] = df[col].fillna(df[col].median())

# Feature Engineering (Age Groups)
df['Age Group'] = np.select(
    [df['Age']<30, (df['Age']>=30) & (df['Age']<=60), df['Age']>60],
    ['Young','Middle Age','Senior'], default='Unknown'
)

# Feature Engineering (Lifestyle Score 0-3)
df['Lifestyle_Score'] = (
    (df['Diet_Quality'] == 'Healthy').astype(int) + 
    (df['Physical_Activity_Level'] == 'High').astype(int) + 
    (df['Stress_Level'] == 'Low').astype(int)
)


## 2. Exploratory Data Analysis (EDA)
### 2.1 Debunking the Myth: Does Gender Matter?
The data proves that the risk is nearly identical between both genders. Diabetes does not discriminate.


In [ ]:
result_gender = df.groupby('Gender')['Diabetes_Risk_Score'].mean().reset_index().round(2)
display(result_gender)

plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='Gender', y='Diabetes_Risk_Score', palette='pastel')
plt.title('Diabetes Risk by Gender (No Significant Difference)')
plt.show()


### 2.2 The Unexpected Truth: Aggressive Early-Onset Risk
Common sense dictates that risk grows with age (and overall, seniors do have a higher average Risk Score). However, diving into the risk groups revealed a striking paradox: 
Young patients who fall into the "High Risk" category record **much more aggressive blood sugar levels** compared to elderly patients in the same category. Early-onset risk strikes harder!


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df,
    x='Age Group',                 
    y='Fasting_Blood_Sugar',       
    hue='Diabetes_Risk',
    order=['Young', 'Middle Age', 'Senior'],
    hue_order=['Low', 'Moderate', 'High'],
    palette='Reds'
)
plt.title("The Unexpected Insight: Aggressive Blood Sugar in High-Risk Youth", fontsize=15)
plt.ylabel("Mean Fasting Blood Sugar")
plt.xlabel("Age Group")
plt.show()


### 2.3 The Hero Insight: Lifestyle vs Genetics
We prove that a patient with a genetic predisposition (DNA) but a perfect life (Lifestyle Score = 3) has a lower risk than someone with no family history who maintains a poor lifestyle (Score = 0).


In [ ]:
plt.figure(figsize=(10, 6))
sns.pointplot(
    data=df,
    x='Lifestyle_Score',
    y='Diabetes_Risk_Score',
    hue='Family_History_Diabetes',
    markers=['o', 's']
)
plt.title("Total Quality of Life (0=Poor, 3=Perfect) vs Genetics", fontsize=15)
plt.ylabel("Diabetes Risk Score")
plt.xlabel("Lifestyle Score (Diet + Exercise + Low Stress)")
plt.show()


### 2.4 Deep Dive: Multivariate Analysis (Heatmap)
Combining Smoking, Genetics, Diet, and Exercise in a single matrix to pinpoint the "dangerous red" zones.


In [ ]:
pivot_result2 = pd.pivot_table(
    data=df,
    values='Diabetes_Risk_Score',
    index=['Smoking_Status','Family_History_Diabetes','Diet_Quality'],
    columns='Physical_Activity_Level',
    aggfunc='mean'
).round(2).sort_values(by='Low', ascending=False)

pivot_clean = pivot_result2[['Unknown','Low','Moderate','High']]

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_clean, annot=True, cmap='YlOrRd', linewidths=0.5)
plt.title("Diabetes Risk Heatmap", fontsize=15)
plt.ylabel("Smoker - Family History - Diet")
plt.xlabel("Physical Activity Level")
plt.show()
